<a href="https://colab.research.google.com/github/quyetttcoder/Fine-tune-LLM-with-small-data/blob/main/vllm_serve.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

In [ ]:
from huggingface_hub import login
login(hf_token)

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
ngrok_token = user_secrets.get_secret("NGROK_TOKEN")

In [ ]:
from pyngrok import ngrok
import os
ngrok.set_auth_token(ngrok_token)

In [ ]:
MY_DOMAIN = "YOUR_NGROK_DOMAIN"

# Lastest qwen3_8b fine tuned: quyetdev/qwen3_8B_v3_fine_tuned_awq
# Latest Llama31_8b fine tuned: quyetdev/llama31_8B_v2_fine_tuned_awq
MODEL_NAME = "quyetdev/qwen3_8B_v3_fine_tuned_awq"

In [ ]:
import subprocess
import os
import time
import requests
import threading


def start_server(gpu_id, port):
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)

    proc = subprocess.Popen(
        [
            "vllm", "serve", MODEL_NAME,
            "--host", "0.0.0.0",
            "--port", str(port),
            # "--max-model-len", "8192",
            "--enable-auto-tool-choice",
            "--tool-call-parser", "hermes",
            "--trust-remote-code"
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=env
    )

    def stream_logs():
        for line in proc.stdout:
            print(f"[GPU {gpu_id}] {line.decode()}", end="")

    threading.Thread(target=stream_logs, daemon=True).start()

    return proc


print("Starting Server on GPU 0 (Port 8000)")
proc1 = start_server(0, 8000)

Starting Server on GPU 0 (Port 8000)


In [ ]:
def wait_for_server(port):
    print(f" Waiting for server on port {port}...")
    for _ in range(60):
        try:
            r = requests.get(f"http://localhost:{port}/health")
            if r.status_code == 200:
                print(f"Server on port {port} is ready!")
                return
        except:
            time.sleep(5)
    raise RuntimeError(f"Server on port {port} failed to start.")

wait_for_server(8000)

 Waiting for server on port 8000...
[GPU 0] (APIServer pid=205) INFO 08-08 02:36:55 [api_utils.py:339] 
[GPU 0] (APIServer pid=205) INFO 08-08 02:36:55 [api_utils.py:339]        █     █     █▄   ▄█
[GPU 0] (APIServer pid=205) INFO 08-08 02:36:55 [api_utils.py:339]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.24.0
[GPU 0] (APIServer pid=205) INFO 08-08 02:36:55 [api_utils.py:339]   █▄█▀ █     █     █     █  model   quyetdev/qwen3_8B_v3_fine_tuned_awq
[GPU 0] (APIServer pid=205) INFO 08-08 02:36:55 [api_utils.py:339]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
[GPU 0] (APIServer pid=205) INFO 08-08 02:36:55 [api_utils.py:339] 
[GPU 0] (APIServer pid=205) INFO 08-08 02:36:55 [api_utils.py:273] non-default args: {'model_tag': 'quyetdev/qwen3_8B_v3_fine_tuned_awq', 'enable_auto_tool_choice': True, 'tool_call_parser': 'hermes', 'host': '0.0.0.0', 'model': 'quyetdev/qwen3_8B_v3_fine_tuned_awq', 'trust_remote_code': True}
[GPU 0] (APIServer pid=205) INFO 08-08 02:37:11 [model.py:598] Resolved architecture: Qwen3ForCa

In [ ]:
tunnel1 = ngrok.connect(8000, url=MY_DOMAIN, pooling_enabled=True)
print(f"Tunnel 1 active: {tunnel1.public_url} -> port 8000")

In [ ]:
!vllm serve "quyetdev/qwen3_8B_fine_tuned_awq" \
    --host 0.0.0.0 \
    --port 8000
    --tensor-parallel-size 2 \
    --gpu-memory-utilization 0.85 \
    --max-model-len 1536